# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path
import logging

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = True
use_refresh_token = False

In [7]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
    else:
        auth_code = os.environ.get("TIKTOK_AUTH_CODE")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

{'access_token': 'ROW_4BYZwwAAAABwUkDEHls0a5HWUXrwXMKTiTad7aZWscGF3YO04dC618tsNg5IyUju9wmyG_yDYWyswxMJmZDUW6fgblohBSVjId5xJjmY2T2ZL6RFLjEx8j22AnX4jTTiS8spO7v1mEk', 'access_token_expire_in': 1785069965, 'refresh_token': 'ROW_ui9aiQAAAAC1Bb7xpqxtsXbX4YggPBqPSJC_n2KwyzjtGQJjqeBiCNIWMJnsgJvu_b0UxC8CJy8', 'refresh_token_expire_in': 4905899553, 'open_id': 'cU4EZQAAAACOV4AEeERJl-yievH_HBssPYsgA5YV2AbEDsJwxBBAMQ', 'seller_name': 'Vitami', 'seller_base_region': 'PH', 'user_type': 0, 'granted_scopes': ['data.bestselling.public.read', 'seller.authorization.info', 'seller.affiliate_collaboration.read', 'seller.affiliate_collaboration.write', 'seller.creator_marketplace.read', 'seller.affiliate_messages.write']}


## Load Creator Data

In [2]:
# load all creators
df_creators_list = pd.read_excel("creators/all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
df_creators = pd.read_csv(CONSOLIDATED_CSV)

df_creators_all = df_creators_list.merge(df_creators, how='left', on="username")

In [23]:
df_creators[df_creators.duplicated(subset=['creator_open_id','username'], keep=False)]

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,video_gmv,live_gmv


In [24]:
# df_creators = df_creators[~(df_creators['creator_open_id'] == 'PC3CtQAAAACOV4AEeERJl-yievH_HBss9ntAFA4yBGy1CxaMuRamZw')]

In [6]:
df_creators_gmv = pd.read_csv("creators/creators_gmv_units_sold_batch1.csv")

In [18]:
df_creators['username'].sort_values()

7164       ..iza.22
1824       ..lhenie
5272      ..luna_35
11562     ..mai..04
19000    ..mikfinds
            ...    
20672           NaN
20748           NaN
20749           NaN
20906           NaN
20936           NaN
Name: username, Length: 20967, dtype: object